# Civic-Link · Colab T4 · 1,000문항 A/B/C

**Drive에 사본을 저장하고 GPU 런타임(T4)을 선택한 뒤 순서대로 실행하세요.**
BGE는 Colab GPU, Qwen 생성·임베딩은 별도 서버를 사용합니다. 전체 1,000문항 × 3조건을 실행합니다.
PC의 VPN은 Colab에 자동 적용되지 않습니다. 서버 연결 점검을 통과해야 진행할 수 있습니다.
Colab Secrets에 `QWEN_ENDPOINT`, `QWEN_EMBEDDING_ENDPOINT`와 필요한 API 키를 등록하세요.
주소는 전체 chat/completions, embeddings URL입니다. 비밀 값은 노트북 코드에 적지 마세요.

[실행·재개 안내](https://github.com/westiion/Civic-Link/blob/main/week8/COLAB.md) ·
[가설·평가 기준](https://github.com/westiion/Civic-Link/blob/main/week8/EXPERIMENT.md)

## 1. Drive와 코드 준비
같은 실험을 재개할 때는 `RUN_NAME`을 유지하세요. 처음 실행한 커밋을 Drive에 보관해 재개에도 사용합니다.
새 코드·설정으로 비교할 때만 새 이름을 지정하세요. 동일 폴더에서 두 노트북을 동시에 실행하지 마세요.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, sys, subprocess, json, getpass

assert sys.version_info >= (3, 12), "Python 3.12 이상 런타임이 필요합니다."
drive.mount('/content/drive')
RUN_NAME = 'week8-t4-1000-v1'
assert RUN_NAME and '/' not in RUN_NAME and '..' not in RUN_NAME
STORAGE = Path('/content/drive/MyDrive/Civic-Link') / RUN_NAME
STORAGE.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/Civic-Link')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/westiion/Civic-Link.git', str(REPO)], check=True)
os.chdir(REPO)
pin = STORAGE / 'code-commit.txt'
if pin.exists():
    commit = pin.read_text().strip()
    assert len(commit) == 40 and all(c in '0123456789abcdef' for c in commit)
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', commit], check=True)
    subprocess.run(['git', 'checkout', '--detach', commit], check=True)
else:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    pin.write_text(commit + '\n')
print('Code:', commit, 'Run:', RUN_NAME)

# Colab의 CUDA PyTorch를 재사용하고 나머지 의존성을 별도 환경에 설치합니다.
VENV = Path('/content/civic-link-venv')
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
PY = str(VENV / 'bin/python')
subprocess.run([PY, '-m', 'pip', 'install', '-r', 'requirements.txt',
                'transformers==4.57.1', 'sentencepiece==0.2.1'], check=True)
def run_script(name, *args):
    subprocess.run([PY, '-u', '-X', 'utf8', 'scripts/' + name, *map(str, args)], check=True)

## 2. GPU·서버 연결 점검
내부망 서버가 Colab에서 닿지 않으면 여기서 중단됩니다. 서버 담당자에게 승인된 접속 경로를 확인한 뒤 다시 실행하세요.
선택 키는 Secrets에 없으면 빈 값으로 처리합니다. 기본 모델 ID가 다르면 아래 모델 이름을 바꾸세요.

In [ ]:
def secret(name, required=False):
    try:
        value = userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        value = ''
    if not value and required:
        value = getpass.getpass(name + ': ').strip()
    if required and not value:
        raise ValueError(name + ' 설정이 필요합니다.')
    if value:
        os.environ[name] = value

for name in ('QWEN_ENDPOINT', 'QWEN_EMBEDDING_ENDPOINT'):
    secret(name, required=True)
for name in ('QWEN_API_KEY', 'QWEN_EMBEDDING_API_KEY', 'QWEN_JUDGE_ENDPOINT', 'QWEN_JUDGE_API_KEY'):
    secret(name)
os.environ['QWEN_MODEL'] = 'Qwen3.8-Flash-Next'
os.environ['QWEN_EMBEDDING_MODEL'] = 'Qwen3-VL-Embedding-8B'

probe = r"""
import sys, os, json, platform, importlib.metadata as metadata
sys.path.insert(0, 'src')
import torch
assert torch.cuda.is_available(), 'CUDA GPU 런타임을 선택하세요. CPU로 자동 전환하지 않습니다.'
from civic_link.agent.client import QwenClient
from civic_link.agent.search import QwenEmbeddingClient
chat = QwenClient(os.environ['QWEN_ENDPOINT'], os.environ['QWEN_MODEL'],
    api_key=os.environ.get('QWEN_API_KEY'), timeout=30, max_tokens=64, json_mode=True)
reply = json.loads(chat.complete([{'role':'user', 'content':'Return only this JSON object: {"ok": true}'}]))
assert reply.get('ok') is True, '생성 서버의 JSON 응답을 확인하세요.'
embedding = QwenEmbeddingClient(os.environ['QWEN_EMBEDDING_ENDPOINT'],
    os.environ['QWEN_EMBEDDING_MODEL'], api_key=os.environ.get('QWEN_EMBEDDING_API_KEY'), timeout=30)
assert len(embedding.embed(['연결 점검'])) == 1
print(json.dumps(dict(python=platform.python_version(), cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0), packages={name:metadata.version(name) for name in
    ('torch','transformers','numpy','kiwipiepy','tokenizers','safetensors','sentencepiece')})))
"""
# 오류의 전체 traceback에 서버 주소가 포함될 수 있어 출력하지 않습니다.
checked = subprocess.run([PY, '-c', probe], capture_output=True, text=True)
if checked.returncode:
    raise RuntimeError('GPU 또는 생성/임베딩 연결 점검 실패. CUDA, 모델 ID, Secrets 권한 및 Colab에서 서버까지의 접속 경로를 확인하세요.')
runtime = json.loads(checked.stdout)
runtime_file = STORAGE / 'runtime.json'
if runtime_file.exists():
    assert json.loads(runtime_file.read_text()) == runtime, '실행 환경이 바뀌었습니다. 원래 환경을 맞추거나 새 RUN_NAME을 사용하세요.'
else:
    runtime_file.write_text(json.dumps(runtime, indent=2) + '\n')
print('GPU·생성·임베딩 연결 확인:', runtime)

## 3. 원문·BGE·색인 준비
BGE의 고정 revision을 다운로드합니다. 정밀도 float32, 입력 한도 1,024토큰, 배치 4를 A/B/C 모두 유지합니다.
색인은 Drive에 저장하며 재개 시 재사용합니다. 색인 생성은 최초에 임베딩 서버를 여러 번 호출합니다.

In [ ]:
run_script('restore_data.py', 'restore')
run_script('download_bge.py')
os.environ['BGE_MODEL_PATH'] = str(REPO / '.cache/reranker-models/models--BAAI--bge-reranker-v2-m3/snapshots/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e')
subprocess.run([PY, '-c', """
import sys, os, math
sys.path.insert(0, 'src')
from civic_link.agent.search import LocalBGEReranker
model = LocalBGEReranker(os.environ['BGE_MODEL_PATH'], device='cuda', batch_size=4, max_length=1024, threads=4)
scores = model.score('대리 신청 서류', ['대리 신청 시 필요한 서류를 안내합니다.', '업무 시간을 안내합니다.'])
assert len(scores) == 2 and all(math.isfinite(s) for s in scores)
assert next(model._model.parameters()).is_cuda
print('BGE CUDA 점검 완료')
"""], check=True)
INDEX = STORAGE / 'index/corpus.npz'
if not INDEX.exists():
    from uuid import uuid4
    # 중간에 끊긴 파일을 완성된 색인으로 잘못 재사용하지 않습니다.
    building = INDEX.with_name('building-' + uuid4().hex + '.npz')
    run_script('build_agent_index.py', '--output', building, '--timeout', 180)
    building.replace(INDEX)
print('색인 준비 완료')

## 4. 전체 1,000문항 A/B/C 실행·재개
`--limit` 없이 실행합니다. 최대 Qwen 8회·도구 6회이며, 공통 첫 검색은 도구 1회에 포함됩니다.
문항별 진행 상황이 출력되고 완료된 조건은 Drive에 저장됩니다. 재실행 시 저장된 결과는 건너뜁니다.
`.running`이 남았다면 이전 실행이 끝났고 다른 노트북도 사용하지 않는지 확인한 뒤 그 잠금 파일만 삭제하세요.

In [ ]:
queries = [json.loads(line) for line in Path('data/benchmark/week8/queries.jsonl').read_text().splitlines() if line.strip()]
assert len(queries) == 1000 and all(set(row) == {'query_id', 'question'} for row in queries)
RUN_DIR = STORAGE / 'comparison'
run_script('run_agent_comparison.py', '--queries', 'data/benchmark/week8/queries.jsonl',
    '--output-dir', RUN_DIR, '--index', INDEX, '--bge-device', 'cuda',
    '--bge-max-length', 1024, '--bge-batch-size', 4, '--bge-threads', 4,
    '--timeout', 180, '--json-mode')

## 5. 근거·비용 평가
완료된 공통 문항과 미완료 문항을 구분합니다. 답변 내용 정확성은 다음 선택 단계를 실행하기 전에는 미평가입니다.
보고서에는 수집/전달/인용 근거, 개선·퇴보, 호출 수와 시간이 포함됩니다.

In [ ]:
from datetime import datetime, timezone
def report_dir(prefix):
    return STORAGE / (prefix + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
EVAL_DIR = report_dir('evidence-eval')
run_script('evaluate_agent_comparison.py', '--run-dir', RUN_DIR,
    '--questions', 'data/benchmark/week8/questions.jsonl', '--output-dir', EVAL_DIR)
print((EVAL_DIR / 'REPORT.md').read_text())
print('보고서 위치:', EVAL_DIR)

## 6. 선택: 사후 내용 평가
**추가 모델 호출이 발생합니다.** `RUN_CONTENT_JUDGE = True`로 바꾸면 실행합니다.
가능하면 별도 평가 모델과 `QWEN_JUDGE_ENDPOINT`를 사용하세요. 아래 기본값은 생성 모델과 같은 모델의 보조 판정이며 독립 검증이 아닙니다.
평가 중 오류가 난 문항은 같은 셀을 재실행해 재시도할 수 있습니다. 개선·퇴보 사례는 사람이 표본 검수하세요.

In [ ]:
RUN_CONTENT_JUDGE = False
JUDGE_MODEL = os.environ['QWEN_MODEL']
if RUN_CONTENT_JUDGE:
    JUDGMENTS = STORAGE / 'judgments'
    run_script('judge_agent_comparison.py', '--run-dir', RUN_DIR,
        '--questions', 'data/benchmark/week8/questions.jsonl', '--output-dir', JUDGMENTS,
        '--model', JUDGE_MODEL, '--timeout', 180, '--json-mode')
    FULL_EVAL = report_dir('full-eval')
    run_script('evaluate_agent_comparison.py', '--run-dir', RUN_DIR,
        '--questions', 'data/benchmark/week8/questions.jsonl', '--judgments', JUDGMENTS,
        '--output-dir', FULL_EVAL)
    print((FULL_EVAL / 'REPORT.md').read_text())
    print('내용 평가 보고서:', FULL_EVAL)